# Data Transformation by Rishab Kumar
## Here the loaded data is 
### - transformed to a high quality data
### - prepared as a part of Silver layer

In [0]:
%sql
use catalog dbacademy;
use schema dbuser_100426;


In [0]:
menu_df_1 = spark.table('bronze_menu_items_csv_file')
menu_df_2 = spark.table('bronze_menu_items_parquet_file')
restaurant_df_1 = spark.table('bronze_restaurant_csv_file')
restaurant_df_2 = spark.table('bronze_restaurant_parquet_file')

In [0]:
#defining the DSFS volume
# dbfs_relative_path = "/Volumes/dbacademy/dbuser_100426/data-volume-swiggy"
# menu_items_csv_file = "/silver_menu_items_csv"
# menu_items_parquet_file = "/silver_menu_items_parquet"
# restaurant_csv_file = "/silver_restaurant_csv"
# restaurant_parquet_file = "/silver_restaurant_parquet"



In [0]:
menu_df_1.printSchema()
menu_df_2.printSchema()


root
 |-- restaurant_id: string (nullable = true)
 |-- restaurant_name: string (nullable = true)
 |-- city_name: string (nullable = true)
 |-- city_slug: string (nullable = true)
 |-- category: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- price: string (nullable = true)
 |-- description: string (nullable = true)
 |-- is_veg: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- rating_count: string (nullable = true)
 |-- in_stock: string (nullable = true)
 |-- has_addons: string (nullable = true)
 |-- has_variants: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)

root
 |-- restaurant_id: string (nullable = true)
 |-- restaurant_name: string (nullable = true)
 |-- city_name: string (nullable = true)
 |-- city_slug: string (nullable = true)
 |-- category: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- price: float (nullabl

In [0]:
# menu_df_1.columns
# menu_df_2.columns
"""
['restaurant_id',
 'restaurant_name',
 'city_name',
 'city_slug',
 'category',
 'item_id',
 'item_name',
 'price',
 'description',
 'is_veg',
 'rating',
 'rating_count',
 'in_stock',
 'has_addons',
 'has_variants',
 'ingestion_time']
 """


"\n['restaurant_id',\n 'restaurant_name',\n 'city_name',\n 'city_slug',\n 'category',\n 'item_id',\n 'item_name',\n 'price',\n 'description',\n 'is_veg',\n 'rating',\n 'rating_count',\n 'in_stock',\n 'has_addons',\n 'has_variants',\n 'ingestion_time']\n "

### Defining the functions 

In [0]:
# a function that takes a dataframe and a name and prints the total number of records and the unique counts of each column
def get_report_for_unique_values(df, name):
    output={}
    print(f"Total records in menu_df_1: {df.count()}")
    print(f"Unique counts of each column in {name}")
    for col in df.columns:
        output[col] = df.select(col).distinct().count()
    return output

In [0]:
# a function that takes a dataframe and a column name and prints the average number of records for each unique value in a column
def density_analysis(df, column_name):
    print(f"Average Items per {column_name}: {df.count()/df.select(column_name).distinct().count()}")



In [0]:
# a function that takes a dataframe and a column name and returns the stats
def get_column_stats(df, column):
    output = {}
    output["distinct"] = df.select(column).distinct().count()
    output["null"] = df.filter(F.col(column).isNull()).count()
    output["max"] = df.select(F.max(column)).collect()[0][0]
    output["min"] = df.select(F.min(column)).collect()[0][0]
    output["avg"] = df.select(F.avg(column)).collect()[0][0]
    return output


In [0]:
import pyspark.sql.functions as F

# a function that takes a dataframe and a column name and converts it to float datatype
def float_conversion(df, column):
    df = df.withColumn(
        column + "_cleaned", F.regexp_replace(column, "[^0-9.]", "")
    )
    df = df.withColumn(
        column + "_float", F.trim(F.col(column + "_cleaned")).try_cast("decimal(10,2)") 
    )
    df = df.drop(column + "_cleaned")
    df = df.drop(column).withColumnRenamed(column + "_float", column)
    df = df.filter(F.col(column).isNotNull())
    return df

# a function that takes a dataframe and a column name and converts it to int datatype
def int_conversion(df, column):
    df = df.withColumn(
        column + "_cleaned", F.regexp_replace(column, "[^0-9]", "")
    )
    df = df.withColumn(
        column + "_int", F.trim(F.col(column + "_cleaned")).try_cast("decimal(20,0)") 
    )
    df = df.drop(column + "_cleaned")
    df = df.drop(column).withColumnRenamed(column + "_int", column)
    df = df.filter(F.col(column).isNotNull())
    return df

# a function that takes a dataframe and a column name and cleans it to id
def id_conversion(df, column):
    df = df.withColumn(
        column + "_cleaned_id", 
        F.when(F.col(column).rlike("^[0-9]+$"), F.col(column))
    )
    df = df.drop(column).withColumnRenamed(column + "_cleaned_id", column)
    df = df.filter(F.col(column).isNotNull())
    return df


def maintain_lower_case(df, columns):
    for column in columns:
        df = df.withColumn(column, F.lower(F.trim(F.col(column))))
    return df

In [0]:
# get_report_for_unique_values(menu_df_1, "menu_df_1")

### The columns and datatypes to be implemented as mentioned below

In [0]:
# enforcing the datatypes as follows
# menu_items.csv / menu_items.parquet
# Column	        Type	Description
# restaurant_id	    string	Foreign key to restaurants table
# restaurant_name	string	Restaurant name
# city_name	        string	City name
# city_slug	        string	URL-friendly city identifier
# category	        string	Menu category (e.g. Recommended, Biryani, Starters > Non Veg)
# item_id	        string	Unique Swiggy menu item identifier
# item_name	        string	Name of the menu item
# price	            float	Price in INR
# description	    string	Item description provided by restaurant
# is_veg	        bool	Whether the item is vegetarian
# rating	        string	Average item rating (if available)
# rating_count	    string	Number of ratings for this item
# in_stock	        bool	Whether the item was in stock at time of scraping
# has_addons	    bool	Whether the item has addon options (sauces, sides, drinks, etc.)
# has_variants	    bool	Whether the item has size/variant options



### Handling duplicates -
### "restaurant_id" and "item_id" 

In [0]:
# cleaning the data with a subset of restaurant_id and item_id
clean_menu_df_1 = menu_df_1.drop_duplicates(subset=["restaurant_id","item_id"])
clean_menu_df_2 = menu_df_2.drop_duplicates(subset=["restaurant_id","item_id"])

In [0]:

density_analysis(clean_menu_df_1, "restaurant_id")

density_analysis(clean_menu_df_2, "restaurant_id")

Average Items per restaurant_id: 67.59432806419065
Average Items per restaurant_id: 70.61258519197219


### Handling - 
### "restaurant_id"

In [0]:
clean_menu_df_1 = id_conversion(clean_menu_df_1, "restaurant_id")
clean_menu_df_2 = id_conversion(clean_menu_df_2, "restaurant_id")

restaurant_id_stats = get_column_stats(clean_menu_df_1, "restaurant_id")

In [0]:
clean_menu_df_1 = id_conversion(clean_menu_df_1, "item_id")
clean_menu_df_2 = id_conversion(clean_menu_df_2, "item_id")

item_id_stats = get_column_stats(clean_menu_df_1, "item_id")

### Handling -
### "price" 

In [0]:
# convert price columns to float datatype
clean_menu_df_1 = float_conversion(clean_menu_df_1, "price").filter("price_float > 1")
clean_menu_df_2 = float_conversion(clean_menu_df_2, "price").filter("price_float > 1")

#get stats for price
price_stats = get_column_stats(clean_menu_df_1, "price")


### Handling -
### "rating", "rating_count"

In [0]:
clean_menu_df_1 = float_conversion(clean_menu_df_1, "rating").filter("rating_float > 0")
clean_menu_df_2 = float_conversion(clean_menu_df_2, "rating").filter("rating_float > 0")

#get stats for rating
rating_stats = get_column_stats(clean_menu_df_1, "rating")


In [0]:
clean_menu_df_1 = int_conversion(clean_menu_df_1, "rating_count").filter("rating_count_int > 0")
clean_menu_df_2 = int_conversion(clean_menu_df_2, "rating_count").filter("rating_count_int > 0")

# get stats for rating_count
rating_count_stats = get_column_stats(clean_menu_df_1, "rating_count")

### Handling - 
### "city_name", "city_slug", "category", "item_name", "description", "is_veg", "in_stock", "has_addons", "has_variants", "restaurant_name"

In [0]:
list_columns = ["city_name", "city_slug", "category", "item_name", "description", "is_veg", "in_stock", "has_addons", "has_variants", "restaurant_name"]
clean_menu_df_1 = maintain_lower_case(clean_menu_df_1, list_columns)
clean_menu_df_2 = maintain_lower_case(clean_menu_df_2, list_columns)

### Reordering the Dataframe

In [0]:

ordered_columns = [
    "restaurant_id",	    
    "restaurant_name",	
    "city_name",   
    "city_slug",  
    "category", 
    "item_id",
    "item_name",
    "price",
    "description",
    "is_veg",
    "rating",
    "rating_count",
    "in_stock",
    "has_addons",
    "has_variants"]

clean_menu_df_1 = clean_menu_df_1.select(ordered_columns)
clean_menu_df_2 = clean_menu_df_2.select(ordered_columns)
    

In [0]:
#performing merge operation on the two "menu_items" dataframes

#adding source column
clean_menu_df_1 = clean_menu_df_1.withColumn("source", F.lit("csv"))
clean_menu_df_2 = clean_menu_df_2.withColumn("source", F.lit("parquet"))

#merging the two dataframes
clean_menu_df_1 = clean_menu_df_1.unionByName(clean_menu_df_2)

#dropping duplicates
clean_menu_df_merged = clean_menu_df_1.dropDuplicates(["restaurant_id", "item_id"])


In [0]:
print(f"Original Dataframe (source: csv) has {menu_df_1.count()} \
    rows and {len(menu_df_1.columns)} columns")
print(f"Cleaned Dataframe (source: parquet) has {clean_menu_df_1.count()} \
    rows and {len(clean_menu_df_1.columns)} columns")

print(f"Original Dataframe (source: parquet) has {menu_df_2.count()} \
    rows and {len(menu_df_2.columns)} columns")
print(f"Cleaned Dataframe (source: parquet) has {clean_menu_df_2.count()} \
    rows and {len(clean_menu_df_2.columns)} columns")

print(f"Merged Dataframe has {clean_menu_df_merged.count()} \
    rows and {len(clean_menu_df_merged.columns)} columns")

display(clean_menu_df_merged.limit(5))

Original Dataframe (source: csv) has 25624046     rows and 16 columns
Cleaned Dataframe (source: parquet) has 4642139     rows and 16 columns
Original Dataframe (source: parquet) has 12800176     rows and 16 columns
Cleaned Dataframe (source: parquet) has 2326155     rows and 16 columns
Merged Dataframe has 2326445     rows and 16 columns


restaurant_id,restaurant_name,city_name,city_slug,category,item_id,item_name,price,description,is_veg,rating,rating_count,in_stock,has_addons,has_variants,source
144188,new york slice pizza,hyderabad,hyderabad,pastas,46370492,mushroom pasta,239.00,choose your sauce.,true,3.60,5,1,true,false,csv
613136,maggi central,hyderabad,hyderabad,fried maggi,101278311,mixed veg fried maggi,100.00,null,true,3.10,6,true,false,false,csv
1170867,deccan elite multi cuisine restaurant,hyderabad,hyderabad,chinese specialities,180226434,chicken 65 [wet],170.00,null,false,3.40,3,true,false,true,csv
35537,sweet truth - cake and desserts,hyderabad,hyderabad,newly launched,180395761,tiramisu cheesecake slice.,219.00,"if coffee is your love language, this tiramisu cheesecake is about to be your new obsession. (energy: 383kcal, carbohydrates: 30gm, proteins: 4gm, fats: 27gm, sodium: 306mg)",true,4.00,1,true,false,false,csv
1201186,1881 dum house: lucknow's legacy,hyderabad,hyderabad,non-veg dum biryani,184592237,chicken lucknowi dum biryani,269.00,"[boneless, half kg, serves 1] boneless chicken in a rich marinade, sealed & slow-cooked with basmati rice.",false,4.70,15,true,true,false,csv


In [0]:
complete_report_clean_menu_df_1 = get_report_for_unique_values(clean_menu_df_merged, "clean_menu_df_merged")

In [0]:

# saving at silver delta tables
# clean_menu_df_merged.write.format("delta").mode("overwrite").saveAsTable("silver_menu_items_table")
# restaurant_df_merged.write.format("delta").mode("overwrite").saveAsTable("silver_restaurants_table")
